In [1]:
!pip install evaluate sacrebleu

import os
import json
import torch
import torch.nn as nn
from torch.optim import AdamW
from tqdm import tqdm
from sacrebleu import corpus_bleu, corpus_chrf
from collections import Counter, defaultdict
from torch.utils.data import Dataset, random_split, DataLoader
from transformers import (
    MBart50Tokenizer,
    MBartConfig,
    MBartForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

import evaluate
import time
from datasets import load_dataset, concatenate_datasets
from sklearn.metrics import accuracy_score, f1_score
import random


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!unzip -o "/content/drive/MyDrive/smol.zip" -d "/content/"

Archive:  /content/drive/MyDrive/smol.zip
  inflating: /content/smol/.git/config  
  inflating: /content/smol/.git/description  
  inflating: /content/smol/.git/FETCH_HEAD  
  inflating: /content/smol/.git/HEAD  
  inflating: /content/smol/.git/hooks/applypatch-msg.sample  
  inflating: /content/smol/.git/hooks/commit-msg.sample  
  inflating: /content/smol/.git/hooks/fsmonitor-watchman.sample  
  inflating: /content/smol/.git/hooks/post-update.sample  
  inflating: /content/smol/.git/hooks/pre-applypatch.sample  
  inflating: /content/smol/.git/hooks/pre-commit.sample  
  inflating: /content/smol/.git/hooks/pre-merge-commit.sample  
  inflating: /content/smol/.git/hooks/pre-push.sample  
  inflating: /content/smol/.git/hooks/pre-rebase.sample  
  inflating: /content/smol/.git/hooks/pre-receive.sample  
  inflating: /content/smol/.git/hooks/prepare-commit-msg.sample  
  inflating: /content/smol/.git/hooks/push-to-checkout.sample  
  inflating: /content/smol/.git/hooks/sendemail-validat

In [4]:
# CONFIG
LANGUAGES = ["es", "lij", "mfe",
             "is", "pcm", "kri",
             "bm", "dyu", "sus",
             "ach", "alz", "luo",
             "sw", "tn", "bem",
             "ks-Deva", "sa", "brx",]

USE_SMOLDOCS = True # May give better results for low-resource languages mBART-50 doesn't know, but not all languages have docs, check documentation.

EPOCHS = 1 # Currently this gives the best results for the time invested.

TRAINING_BATCH_SIZE = 8 # Saves memory.

LIMIT_TEST_SIZES = False # Limits the size of test and val split to 173 / 86.

smol_path = "/content/smol/"

Typology-aware mBART Model

In [5]:
class TypologyAwareMBart(nn.Module):
    def __init__(self, base_model, script_classes, family_classes, region_classes):
        super().__init__()
        self.model = base_model
        self.hidden = base_model.config.d_model

        feature_dim = self.hidden // 4

        self.script_emb = nn.Embedding(script_classes, feature_dim)
        self.family_emb = nn.Embedding(family_classes, feature_dim)
        self.region_emb = nn.Embedding(region_classes, feature_dim)

        self.proj = nn.Linear(3 * feature_dim, self.hidden)

    def forward(self, input_ids, attention_mask, labels,
                script_ids, family_ids, region_ids):

        batch_size, seq_len = input_ids.size()

        # typology embeddings
        s = self.script_emb(script_ids).squeeze(1)
        f = self.family_emb(family_ids).squeeze(1)
        r = self.region_emb(region_ids).squeeze(1)

        typology_vec = torch.cat([s, f, r], dim=-1)
        typology_vec = self.proj(typology_vec)
        typology_tokens = typology_vec.unsqueeze(1).expand(batch_size, seq_len, self.hidden)

        inputs_embeds = self.model.model.encoder.embed_tokens(input_ids)
        inputs_embeds = inputs_embeds + typology_tokens

        return self.model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels
        )
    def generate( self, input_ids, attention_mask, script_ids, family_ids,
                 region_ids, forced_bos_token_id=None, max_length=128, **kwargs):
        batch_size, seq_len = input_ids.size()

        s = self.script_emb(script_ids).squeeze(1)
        f = self.family_emb(family_ids).squeeze(1)
        r = self.region_emb(region_ids).squeeze(1)

        typology_vec = torch.cat([s, f, r], dim=-1)
        typology_vec = self.proj(typology_vec)
        typology_tokens = typology_vec.unsqueeze(1).expand(
            batch_size, seq_len, self.hidden
        )

        inputs_embeds = (
            self.model.model.encoder.embed_tokens(input_ids)
            + typology_tokens
        )

        return self.model.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            forced_bos_token_id=forced_bos_token_id,
            max_length=max_length,
            **kwargs
        )


In [6]:
SCRIPT_MAP = {"Latin": 0, "Devanagari": 1}
FAMILY_MAP = {
    "Germanic": 0, "Italic": 1, "Central-Mande": 2,
    "Indo-Aryan": 3, "Lwoo": 4, "East-Bantu": 5,
}
REGION_MAP = {"Europe": 0, "Africa": 1, "SouthAsia": 2}

LANG_META = {
    "en": {"script": "Latin", "family": "Germanic", "region": "Europe"},
    "es": {"script": "Latin", "family": "Italic", "region": "Europe"},
    "lij": {"script": "Latin", "family": "Italic", "region": "Europe"},
    "mfe": {"script": "Latin", "family": "Italic", "region": "Africa"},
    "is": {"script": "Latin", "family": "Germanic", "region": "Europe"},
    "pcm": {"script": "Latin", "family": "Germanic", "region": "Africa"},
    "kri": {"script": "Latin", "family": "Germanic", "region": "Africa"},
    "ks-Deva": {"script": "Devanagari", "family": "Indo-Aryan", "region": "SouthAsia"},
    "sa": {"script": "Devanagari", "family": "Indo-Aryan", "region": "SouthAsia"},
    "brx": {"script": "Latin", "family": "Indo-Aryan", "region": "SouthAsia"},
    "bm": {"script": "Latin", "family": "Central-Mande", "region": "Africa"},
    "dyu": {"script": "Latin", "family": "Central-Mande", "region": "Africa"},
    "sus": {"script": "Latin", "family": "Central-Mande", "region": "Africa"},
    "ach": {"script": "Latin", "family": "Lwoo", "region": "Africa"},
    "alz": {"script": "Latin", "family": "Lwoo", "region": "Africa"},
    "luo": {"script": "Latin", "family": "Lwoo", "region": "Africa"},
    "sw": {"script": "Latin", "family": "East-Bantu", "region": "Africa"},
    "tn": {"script": "Latin", "family": "East-Bantu", "region": "Africa"},
    "bem": {"script": "Latin", "family": "East-Bantu", "region": "Africa"},
}

Data loading

In [7]:
# ===================== HELPER FUNCTIONS =====================================#

# To figure out what languages have smoldocs
def try_load_jsonl(path):
    if os.path.exists(path):
        return load_dataset("json", data_files=path)["train"]
    return None

# Load all the datasets
def load_language_datasets(lang):
    gatitos = try_load_jsonl(f"{smol_path}/gatitos/en_{lang}.jsonl")
    smolsent = try_load_jsonl(f"{smol_path}/smolsent/en_{lang}.jsonl")
    smoldoc = None

    if USE_SMOLDOCS:
        smoldoc = try_load_jsonl(f"{smol_path}/smoldoc/en_{lang}.jsonl")

    # Normalize gatitos ("trgs" -> "trg")
    if gatitos is not None:
        gatitos = gatitos.map(unify_gatitos)

    # Flatten smoldoc to smolsent
    if smoldoc is not None:
        smoldoc_flat = smoldoc.map(flatten_smoldoc_to_smolsent, batched=True, remove_columns=smoldoc.column_names)
        smolsent = concatenate_datasets([smolsent, smoldoc_flat])

    return gatitos, smolsent

# Convert 'trgs' list -> single 'trg' string for gatitos
def unify_gatitos(example):
    example["trg"] = example["trgs"][0]
    return example

# Convert smoldoc 'srcs' and 'trgs' into 'src' and 'trg'
def flatten_smoldoc_to_smolsent(batch):
    new_src = []
    new_trg = []
    new_tl  = []

    for srcs, trgs, tl in zip(batch["srcs"], batch["trgs"], batch["tl"]):

        # ensure srcs/trgs match
        if len(srcs) != len(trgs):
            print("MISMATCH BETWEEN TRGS AND SRC")

        for s, t in zip(srcs, trgs):
            new_src.append(s)
            new_trg.append(t)
            new_tl.append(tl)

    if len(new_src) > 0:
        print(f"Total entries: {len(new_src)}")
        print("First flattened entry:")
        print(f"  src: {new_src[1]}")
        print(f"  trg: {new_trg[1]}")
        print(f"  tl:  {new_tl[1]}")
        print("-" * 50)

    return {
        "src": new_src,
        "trg": new_trg,
        "tl":  new_tl
    }

# Create the train, test, and val split
def split_smolsent(ds):
    train_test = ds.train_test_split(test_size=0.30, seed=42)
    train_ds = train_test["train"]
    test_val_ds = train_test["test"]

    val_test = test_val_ds.train_test_split(test_size=2/3, seed=42)
    val_ds = val_test["train"]
    test_ds = val_test["test"]

    return train_ds, val_ds, test_ds

# This way we ensure that we get the same number of test/val entries per language.
def split_smolsent_maxSize(ds, maxTestSize=173, maxValSize=86):
  seed = 42
  # Ensure reproducibility
  random.seed(seed)

  # Total indices
  all_indices = list(range(len(ds)))
  random.shuffle(all_indices)

  # Slice indices
  test_indices = all_indices[:maxTestSize]
  val_indices  = all_indices[maxTestSize:maxTestSize + maxValSize]
  train_indices = all_indices[maxTestSize + maxValSize:]

  # Create subsets
  test_ds  = ds.select(test_indices)
  val_ds   = ds.select(val_indices)
  train_ds = ds.select(train_indices)

  return train_ds, val_ds, test_ds

# Create the special language token corresponding to a language
def create_language_ID(lang):
    return f"{lang}_XX"

In [15]:
#========================== MAIN FUNCTION LOOP ===============================#
def add_typology_metadata(dataset):
    """
    Adds script_ids, family_ids, and region_ids columns based on 'tl'.
    Assumes LANG_META, SCRIPT_MAP, FAMILY_MAP, REGION_MAP are defined.
    """
    def map_meta(example):
        meta = LANG_META[example["tl"]]
        return {
            "script_ids": SCRIPT_MAP[meta["script"]],
            "family_ids": FAMILY_MAP[meta["family"]],
            "region_ids": REGION_MAP[meta["region"]],
        }

    return dataset.map(map_meta)

# Temporary structures
training_sets = []
gatitos_sets = []
val_sets = []

# The language key mapped to test set
test_dataset = {}

for lang in LANGUAGES:
  print(f"Processing language: {lang}")

  gatitos, smolsent = load_language_datasets(lang)

  # Remove other columns
  columns_to_keep = ["src", "trg", "tl"]

  if gatitos is not None:
      gatitos = gatitos.select_columns(columns_to_keep)
      gatitos_sets.append(gatitos)

  if smolsent is not None:
      smolsent = smolsent.select_columns(columns_to_keep)

  train, val, test = split_smolsent(smolsent)
  if LIMIT_TEST_SIZES:
    train, val, test = split_smolsent_maxSize(smolsent)

  training_sets.append(train)
  val_sets.append(val)

  test_dataset[lang] = test

# Merge sets into final
train_dataset = concatenate_datasets(training_sets + gatitos_sets)
val_dataset = concatenate_datasets(val_sets)
print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))
print("Test sizes:", {k: len(v) for k, v in test_dataset.items()})


Processing language: es
Processing language: lij
Processing language: mfe
Processing language: is
Processing language: pcm
Processing language: kri
Processing language: bm
Processing language: dyu
Processing language: sus
Processing language: ach
Processing language: alz
Processing language: luo
Processing language: sw
Processing language: tn
Processing language: bem
Processing language: ks-Deva
Processing language: sa
Processing language: brx
Train size: 90886
Validation size: 3857
Test sizes: {'es': 173, 'lij': 338, 'mfe': 338, 'is': 173, 'pcm': 495, 'kri': 495, 'bm': 800, 'dyu': 338, 'sus': 338, 'ach': 338, 'alz': 338, 'luo': 800, 'sw': 1567, 'tn': 338, 'bem': 338, 'ks-Deva': 173, 'sa': 173, 'brx': 173}


Load model

In [9]:
# Load model
tokenizer = MBart50Tokenizer.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
base_model = MBartForConditionalGeneration.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")

# Setup language tokens that don't exist in mBART
special_tokens = [f"<{lang}_XX>" for lang in LANGUAGES]
tokenizer.add_tokens(special_tokens, special_tokens=True)
base_model.resize_token_embeddings(len(tokenizer))

# Map language "es" to token ID "es_XX"
LANG_MAP = {lang: create_language_ID(lang) for lang in LANGUAGES}
print(LANG_MAP)

print("Building Typology-Aware mBART...")
model = TypologyAwareMBart(
    base_model,
    script_classes=len(SCRIPT_MAP),
    family_classes=len(FAMILY_MAP),
    region_classes=len(REGION_MAP)
)
# Detect GPU or use CPU
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Using device:", device)

model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


{'es': 'es_XX', 'lij': 'lij_XX', 'mfe': 'mfe_XX', 'is': 'is_XX', 'pcm': 'pcm_XX', 'kri': 'kri_XX', 'bm': 'bm_XX', 'dyu': 'dyu_XX', 'sus': 'sus_XX', 'ach': 'ach_XX', 'alz': 'alz_XX', 'luo': 'luo_XX', 'sw': 'sw_XX', 'tn': 'tn_XX', 'bem': 'bem_XX', 'ks-Deva': 'ks-Deva_XX', 'sa': 'sa_XX', 'brx': 'brx_XX'}
Building Typology-Aware mBART...
Using device: cuda


TypologyAwareMBart(
  (model): MBartForConditionalGeneration(
    (model): MBartModel(
      (shared): MBartScaledWordEmbedding(250072, 1024, padding_idx=1)
      (encoder): MBartEncoder(
        (embed_tokens): MBartScaledWordEmbedding(250072, 1024, padding_idx=1)
        (embed_positions): MBartLearnedPositionalEmbedding(1026, 1024)
        (layers): ModuleList(
          (0-11): 12 x MBartEncoderLayer(
            (self_attn): MBartAttention(
              (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
            )
            (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (activation_fn): ReLU()
            (fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (fc2)

Training

In [11]:
def preprocess_function(batch):
    # Tokenize source
    model_inputs = tokenizer(batch["src"], max_length=128, truncation=True)

    # Map tl -> BOS token ID correctly
    bos_ids = []
    script_ids = []
    family_ids = []
    region_ids = []

    for tl in batch["tl"]:
        lang_token_str = LANG_MAP[tl] # e.g., "es_XX"
        # Get the token ID for the special language token
        bos_ids.append(tokenizer.convert_tokens_to_ids(f"<{lang_token_str}>"))

        # Typology metadata
        meta = LANG_META[tl]
        script_ids.append(SCRIPT_MAP[meta["script"]])
        family_ids.append(FAMILY_MAP[meta["family"]])
        region_ids.append(REGION_MAP[meta["region"]])

    # Tokenize target
    target_ids = [
        tokenizer.encode(t, truncation=True, max_length=124, add_special_tokens=False)
        for t in batch["trg"]
    ]

    # Prepend BOS
    labels = []
    for bos, seq in zip(bos_ids, target_ids):
        labels.append([bos] + seq + [tokenizer.eos_token_id])

    model_inputs["labels"] = labels

    # Inject metadata into batch
    model_inputs["script_ids"] = script_ids
    model_inputs["family_ids"] = family_ids
    model_inputs["region_ids"] = region_ids

    # Keep original target text in the processed dataset for evaluation references
    model_inputs["trg"] = batch["trg"]

    return model_inputs

# tokenize everything
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=["src", "tl"])
tokenized_val   = val_dataset.map(preprocess_function, batched=True, remove_columns=["src", "tl"])

Map:   0%|          | 0/90886 [00:00<?, ? examples/s]

Map:   0%|          | 0/3857 [00:00<?, ? examples/s]

In [12]:



# Collator for some padding
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Args
training_args = Seq2SeqTrainingArguments(
    output_dir="finetuned-mBart-ES",

    per_device_train_batch_size=TRAINING_BATCH_SIZE,
    per_device_eval_batch_size=16,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,

    learning_rate=3e-5,

    #label_smoothing_factor=0.1,
    save_safetensors=False,
    save_strategy="epoch",
    predict_with_generate=True,
    logging_strategy="steps",
    logging_steps=20,

    warmup_ratio=0.1, # May be removed?

    report_to="none"
)

# Define the trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Train!
trainer.train()

/tmp/ipython-input-2079841395.py:28: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss
20,8.700400
40,8.477700
60,8.255800
80,7.764900
100,7.687700
120,7.547300
140,6.993800
160,7.150700
180,7.033600
200,7.019600


TrainOutput(global_step=11361, training_loss=3.6238406028517556, metrics={'train_runtime': 1426.5262, 'train_samples_per_second': 63.711, 'train_steps_per_second': 7.964, 'total_flos': 0.0, 'train_loss': 3.6238406028517556, 'epoch': 1.0})

In [13]:
trainer.save_model("my_mbart_model")
tokenizer.save_pretrained("my_mbart_model")

('my_mbart_model/tokenizer_config.json',
 'my_mbart_model/special_tokens_map.json',
 'my_mbart_model/sentencepiece.bpe.model',
 'my_mbart_model/added_tokens.json')

Evaluation

In [16]:
tokenized_test_datasets = {
    lang: dataset.map(preprocess_function, batched=True, remove_columns=["src", "tl"])
    for lang, dataset in test_dataset.items()
}

class CustomDataCollator:
    def __init__(self, tokenizer, model):
        self.data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

    def __call__(self, features):
        # features is a list of dicts, e.g., [{'input_ids': ..., 'trg': 'text1'}, ...]
        # Separate 'trg' (original target references) from the other features
        trg_references = [f.pop('trg') for f in features] # Pop removes 'trg' from the dicts

        # Now, pass the modified features to the Hugging Face data collator
        collated_batch = self.data_collator(features)

        # Add back the original 'trg' references under a new key
        collated_batch['trg_references'] = trg_references

        return collated_batch

def evaluate_translation_model(model, tokenizer, tokenized_test_datasets, batch_size=16):
    """
    Evaluate TypologyAwareMBart on multiple languages using pre-tokenized datasets.

    Args:
        model: TypologyAwareMBart model
        tokenizer: HuggingFace tokenizer
        tokenized_test_datasets: dict of {lang_code: preprocessed Dataset}
        batch_size: batch size for DataLoader

    Returns:
        results: dict of {lang_code: {"bleu": float, "chrf": float}}
    """
    # Load metrics
    bleu_metric = evaluate.load("sacrebleu")
    chrf_metric = evaluate.load("chrf")

    model.eval()
    device = next(model.parameters()).device
    results = {}

    for lang, dataset in tokenized_test_datasets.items():
        print(f"Evaluating language: {lang}")

        # Use the custom collator
        custom_collator = CustomDataCollator(tokenizer, model.model)
        loader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=False,
            collate_fn=custom_collator
        )

        preds = []
        refs = []

        for batch in tqdm(loader):
            # Move input tensors to device
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            script_ids = torch.tensor(batch["script_ids"]).to(device)
            family_ids = torch.tensor(batch["family_ids"]).to(device)
            region_ids = torch.tensor(batch["region_ids"]).to(device)

            # Determine forced BOS token
            tgt_lang_code = LANG_MAP[lang]
            # FIX: Use convert_tokens_to_ids for custom special tokens, including the angle brackets.
            bos_id = tokenizer.convert_tokens_to_ids(f"<{tgt_lang_code}>")

            # Generate translations
            with torch.no_grad():
                generated_ids = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    script_ids=script_ids,
                    family_ids=family_ids,
                    region_ids=region_ids,
                    forced_bos_token_id=bos_id,
                    max_length=128
                )

            # Decode predictions
            decoded_preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
            decoded_preds = [p.strip() for p in decoded_preds]

            # References: Use the original 'trg' column directly for evaluation
            decoded_refs = [t.strip() for t in batch["trg_references"]]

            preds.extend(decoded_preds)
            refs.extend([[r] for r in decoded_refs])

        # Compute metrics
        bleu = bleu_metric.compute(predictions=preds, references=refs)
        chrf = chrf_metric.compute(predictions=preds, references=refs)

        print(f"{lang} BLEU: {bleu['score']:.2f}, chrF: {chrf['score']:.2f}")
        results[lang] = {"bleu": bleu["score"], "chrf": chrf["score"]}

    return results

# Evaluate all languages
bleu_scores = evaluate_translation_model(model, tokenizer, tokenized_test_datasets, batch_size=16)

Evaluating language: es


  0%|          | 0/11 [00:00<?, ?it/s]/tmp/ipython-input-3956766101.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  script_ids = torch.tensor(batch["script_ids"]).to(device)
/tmp/ipython-input-3956766101.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  family_ids = torch.tensor(batch["family_ids"]).to(device)
/tmp/ipython-input-3956766101.py:65: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  region_ids = torch.tensor(batch["region_ids"]).to(device)
  9%|▉         | 1/11 [00:00<00:07,  1.26it/s]/tmp/ipython-input-3956766101.py:63: 

es BLEU: 30.22, chrF: 56.87
Evaluating language: lij


  0%|          | 0/22 [00:00<?, ?it/s]/tmp/ipython-input-3956766101.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  script_ids = torch.tensor(batch["script_ids"]).to(device)
/tmp/ipython-input-3956766101.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  family_ids = torch.tensor(batch["family_ids"]).to(device)
/tmp/ipython-input-3956766101.py:65: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  region_ids = torch.tensor(batch["region_ids"]).to(device)
  9%|▉         | 2/22 [00:02<00:21,  1.09s/it]/tmp/ipython-input-3956766101.py:63: 

lij BLEU: 12.55, chrF: 39.01
Evaluating language: mfe


...

ks-Deva BLEU: 0.73, chrF: 12.83
Evaluating language: sa


  0%|          | 0/11 [00:00<?, ?it/s]/tmp/ipython-input-3956766101.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  script_ids = torch.tensor(batch["script_ids"]).to(device)
/tmp/ipython-input-3956766101.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  family_ids = torch.tensor(batch["family_ids"]).to(device)
/tmp/ipython-input-3956766101.py:65: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  region_ids = torch.tensor(batch["region_ids"]).to(device)
  9%|▉         | 1/11 [00:01<00:10,  1.06s/it]/tmp/ipython-input-3956766101.py:63: 

sa BLEU: 0.56, chrF: 25.98
Evaluating language: brx


  0%|          | 0/11 [00:00<?, ?it/s]/tmp/ipython-input-3956766101.py:63: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  script_ids = torch.tensor(batch["script_ids"]).to(device)
/tmp/ipython-input-3956766101.py:64: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  family_ids = torch.tensor(batch["family_ids"]).to(device)
/tmp/ipython-input-3956766101.py:65: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  region_ids = torch.tensor(batch["region_ids"]).to(device)
  9%|▉         | 1/11 [00:01<00:18,  1.86s/it]/tmp/ipython-input-3956766101.py:63: 

brx BLEU: 0.27, chrF: 13.11
